# 06.08_All_seurat_Integrate_R

Seurat 多方法整合探索。

- 当前文件：`analysis/06_single_cell_analysis/06.08_All_seurat_Integrate_R.ipynb`
- 原始来源：`Codes/06.08_R_seurat_Integrate.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`Matrix`, `Seurat`, `ggplot2`, `harmony`, `scales`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。

**本文件说明：** 多方法探索仍包含交互式对象依赖，例如 objs；06.09 为独立 RPCA 分支。


r_base

## Seurat unintegrated

In [ ]:
library(Seurat)
library(ggplot2)

In [ ]:
# 1. 路径与物种设置
ogs_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/"
species_order <- c("Dare", "Neve", "Clhe", "Auco", "TrH1", "TrH2", "HoH13", "ClH23", "Spla")
# 定义门类映射关系
phylum_map <- c(
    "Spla"  = "Porifera",
    "ClH23" = "Placozoa", "HoH13" = "Placozoa", "TrH2" = "Placozoa", "TrH1" = "Placozoa",
    "Auco"  = "Cnidaria", "Clhe"  = "Cnidaria", "Neve" = "Cnidaria",
    "Dare"  = "Chordata"
)

In [ ]:
# 2. 数据读取与元数据注入
message(">>> 正在读取 RDS 文件并注入标签...")
objs <- lapply(species_order, function(sp) {
    obj <- readRDS(paste0(ogs_path, sp, ".OG.normalized.rds"))
    
    # [核心修正点] 使用 unname() 去掉 "Dare =" 这种名称干扰
    # 这样赋给 obj$Phylum 的就是一个纯粹的字符串，Seurat 会自动将其广播到所有细胞
    obj$species <- sp
    obj$Phylum  <- unname(phylum_map[sp])
    
    # 设置因子顺序
    obj$Phylum <- factor(obj$Phylum, levels = c("Porifera", "Placozoa", "Cnidaria", "Chordata"))
    
    return(obj)
})

# 提取并计算共有 OG 集合
ogs_list <- lapply(objs, rownames)
common_ogs <- Reduce(intersect, ogs_list)
message(paste(">>> 共有 OG 数量:", length(common_ogs)))

In [ ]:
# 3. 统一子集化 (只保留共有基因)
message(">>> 正在进行基因子集化...")
objs <- lapply(objs, function(obj) {
  return(subset(obj, features = common_ogs))
})

In [ ]:
# 4. 数据合并 (Metadata 将会被自动整合)
message(">>> 正在合并对象...")
# add.cell.ids 确保 Barcode 唯一性
merged_raw <- merge(objs[[1]], y = objs[-1], add.cell.ids = species_order)

# 释放原始列表内存
rm(objs); gc()

# 验证合并后的标签是否准确
message(">>> 合并完成，验证标签分布：")
table(merged_raw$species, merged_raw$Phylum)

In [ ]:
# 5. 标准分析流程
message(">>> 正在进行预处理与降维...")
DefaultAssay(merged_raw) <- "RNA"

# 在共有基因池中寻找变异基因
merged_raw <- FindVariableFeatures(merged_raw, selection.method = "vst", nfeatures = 2000)

# 归一化与缩放
# 注意：ScaleData 对 VariableFeatures 进行缩放以加速计算
merged_raw <- ScaleData(merged_raw, features = VariableFeatures(merged_raw))

# PCA 降维
merged_raw <- RunPCA(merged_raw, npcs = 50, verbose = FALSE)

# UMAP 降维与聚类
merged_raw <- RunUMAP(merged_raw, dims = 1:30, verbose = FALSE)
merged_raw <- FindNeighbors(merged_raw, dims = 1:30, verbose = FALSE)
merged_raw <- FindClusters(merged_raw, resolution = 0.5, verbose = FALSE)

In [ ]:
# 6. 设置全局统一配色 (基于您要求的门类色系)
# 多孔:黄; 扁盘:红; 刺胞:蓝; 斑马鱼:绿
species_colors <- c(
    # 多孔动物 (Porifera)
    "Spla"  = "#fba414",
    
    # 扁盘动物 (Placozoa)
    "ClH23" = "#ffa8a7",
    "HoH13" = "#eb7f7f",
    "TrH2"  = "#ff5d4e",
    "TrH1"  = "#EC2B24",
    
    # 刺胞动物 (Cnidaria)
    "Auco"  = "#2A52BE",
    "Clhe"  = "#4374B3",
    "Neve"  = "#6DA0E2",
    
    # 脊索动物 (Chordata - 斑马鱼)
    "Dare"  = "#43b244"
)
# 设置门类配色方案 (对应四大色系)
phylum_colors <- c(
    "Porifera" = "#fba414", # 黄色
    "Placozoa" = "#EC2B24", # 红色
    "Cnidaria" = "#2A52BE", # 蓝色
    "Chordata" = "#43b244"  # 绿色
)

In [ ]:
# 7. 绘图与保存
message(">>> 正在生成可视化图表...")

# A. 按物种查看 (检查物种效应)
p1 <- DimPlot(merged_raw, group.by = "species", cols = species_colors, pt.size = 0.5, raster = FALSE) +
  theme_minimal() +
  ggtitle("Global UMAP by Species (9 Species)")

# B. 按聚类查看
p2 <- DimPlot(merged_raw, group.by = "seurat_clusters", label = TRUE, pt.size = 0.5, raster = FALSE) +
  theme_minimal() +
  ggtitle("Global UMAP by Clusters")

# C. 联动查看 (Split by species)
p3 <- DimPlot(merged_raw, group.by = "species", split.by = "species", cols = species_colors, ncol = 3)

# D. 按物种查看 (检查物种效应)
p4 <- DimPlot(merged_raw, group.by = "Phylum", cols = phylum_colors, pt.size = 0.5, raster = FALSE) +
  theme_minimal() +
  ggtitle("Global UMAP by Phylum (4 Phylum)")

# 打印查看
print(p1)
print(p2)
print(p3)
print(p4)

In [ ]:

# 保存对象
# saveRDS(merged_raw, file = "Merged_9Species_Unintegrated.rds")

message(">>> 分析完成！")

In [ ]:
species_orders <- c("Dare", "Neve", "Clhe", "Auco", "TrH1", "TrH2", "HoH13", "ClH23", "Spla")
species_colors <- c(
    "Spla"="#fba414", "ClH23"="#ffa8a7", "HoH13"="#eb7f7f", 
    "TrH2"="#ff5d4e", "TrH1"="#EC2B24", "Auco"="#2A52BE", 
    "Clhe"="#4374B3", "Neve"="#6DA0E2", "Dare"="#43b244"
)

p_unintegrate <- DimPlot(
    merged_raw, 
    group.by = "species", 
    cols = species_colors, 
    # pt.size = 0.5, 
    raster = TRUE) + 
    # 去除横纵坐标轴
    theme(
        axis.title = element_blank(),   # 去除轴标题
        axis.text = element_blank(),    # 去除轴刻度文字
        axis.ticks = element_blank(),   # 去除轴刻度线
        axis.line = element_blank(),    # 去除轴线
        # 图例按特定顺序排列
        legend.position = "right",      # 图例位置
        legend.direction = "vertical",  # 垂直排列
        legend.text = element_text(size = 12),
        legend.title = element_text(size = 14, face = "bold")
    ) +
    # 手动设置图例顺序（关键部分）
    scale_color_manual(
        values = species_colors,
        breaks = species_orders
    )
p_unintegrate

## Seurat RPCA (no counts required)

In [ ]:
library(Seurat)
library(Matrix)
set.seed(42)

In [ ]:
DefaultAssay(objs[[1]])
Assays(objs[[1]])
slotNames(objs[[1]][["RNA"]])

In [ ]:
# 先对每个对象做 PCA
objs_rpca <- lapply(objs, function(x) {
  x <- ScaleData(x, features = common_ogs, verbose = FALSE)
  x <- RunPCA(x, features = common_ogs, npcs = 50, verbose = FALSE)
  x
})

In [ ]:
# 然后用 reduction = "rpca"
anchors_rpca <- FindIntegrationAnchors(
  object.list = objs_rpca,
  anchor.features = common_ogs,
  reduction = "rpca",   # <—— 改成 rpca
  dims = 1:50
)

In [ ]:
# 数据整合
integrated_rpca <- IntegrateData(anchorset = anchors_rpca, dims = 1:50)

In [ ]:
# ---------- 下游降维与聚类 ----------
DefaultAssay(integrated_rpca) <- "integrated"
integrated_rpca <- ScaleData(integrated_rpca, verbose = FALSE)
integrated_rpca <- RunPCA(integrated_rpca, npcs = 50, verbose = FALSE)
integrated_rpca <- RunUMAP(integrated_rpca, dims = 1:50, verbose = FALSE)
integrated_rpca <- FindNeighbors(integrated_rpca, dims = 1:50, verbose = FALSE)
integrated_rpca <- FindClusters(integrated_rpca, resolution = 0.5)

In [ ]:
integrated_rpca

In [ ]:
head(integrated_rpca@meta.data)

In [ ]:
# ---------- 7) 可视化 ----------
p_by_species  <- DimPlot(integrated_rpca, reduction = "umap", group.by = "species")
p_by_clusters <- DimPlot(integrated_rpca, reduction = "umap", group.by = "seurat_clusters", label = TRUE)

print(p_by_species); print(p_by_clusters)

In [ ]:
# ---------- 8) 保存结果 ----------
# 可选：导出嵌入与元数据（便于后续评估/作图）
dir.create("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA", showWarnings = FALSE, recursive = TRUE)
saveRDS(integrated_rpca, file = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA/integrated_rpca.rds")
write.csv(Embeddings(integrated_rpca, "pca"),
          "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA/pca_embeddings.csv")
write.csv(Embeddings(integrated_rpca, "umap"),
          "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA/umap_embeddings.csv")
write.csv(integrated_rpca@meta.data,
          "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/SingleCellIntegrated/Seurat_RPCA/metadata.csv")

In [ ]:
target_types <- c("Auco_9", "Auco_14", "NPC", "Neural", "neuronal", "peptidergic", "Peptidocytes")
cells_use <- WhichCells(integrated_rpca, expression = CellType %in% target_types)

DimPlot(
  integrated_rpca,
  group.by = "CellType",
  cells.highlight = cells_use,
  cols.highlight = "red",     # 或c("red","blue",...)自定义多色
  cols = "gray90"
)
# +
#   ggtitle("Highlight selected CellTypes")


## Seurat Harmony

In [ ]:
library(Seurat)
library(harmony)

In [ ]:
ogs_path <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/scOrthoGeneRDS/"
species <- c("Dare", "Neve", "Auco", "Clhe", "TrH1", "TrH2", "HoH13", "ClH23", "Spla")
objs <- lapply(species, function(sp) readRDS(paste0(ogs_path, sp, ".OG.normalized.rds")))

# 提取每个对象的基因（行名/OG名）
ogs_list <- lapply(objs, function(obj) rownames(obj))

In [ ]:
# 得到所有对象共有的OG集合
common_ogs <- Reduce(intersect, ogs_list)
length(common_ogs)

In [ ]:
# 统一子集（确保只包含共有OG）
objs <- lapply(objs, function(obj) {
  subset(obj, features = common_ogs)
})

In [ ]:
# 添加物种标签（orig.ident、species等）
for (i in seq_along(objs)) {
  objs[[i]]$species <- species[i]
}

In [ ]:
DefaultAssay(objs[[1]])
Assays(objs[[1]])
slotNames(objs[[1]][["RNA"]])


In [ ]:
# 合并所有物种对象为一个大Seurat对象
combined <- objs[[1]]
if (length(objs) > 1) {
  for (i in 2:length(objs)) {
    combined <- merge(combined, y = objs[[i]])
  }
}

In [ ]:
# 选高变基因
combined <- FindVariableFeatures(combined, selection.method = "vst", nfeatures = 2000)

# 标准化
combined <- ScaleData(combined, features = VariableFeatures(combined))

# PCA
combined <- RunPCA(combined, features = VariableFeatures(combined), npcs = 30)


In [ ]:
head(combined@meta.data)

In [ ]:
# Harmony整合
# 关键：group.by.vars 指定物种标签列名，默认为 "species"
# 默认用PCA空间
combined <- RunHarmony(
  object = combined,
  group.by.vars = "species"
)

In [ ]:
# UMAP与聚类
combined <- RunUMAP(combined, reduction = "harmony", dims = 1:30)
combined <- FindNeighbors(combined, reduction = "harmony", dims = 1:30)
combined <- FindClusters(combined, resolution = 0.5)

In [ ]:
# 按物种分色
DimPlot(combined, reduction = "umap", group.by = "species", label = TRUE)

In [ ]:
# 按物种分色
DimPlot(combined, reduction = "umap", group.by = "seurat_clusters", label = TRUE)

In [ ]:
table(combined$species)

In [ ]:
DimPlot(combined, group.by = "species", split.by = "species", ncol = 3)

In [ ]:
target_types <- c("Auco_9", "Auco_14", "Neural", "neuronal", "peptidergic", "Peptidocytes")

# 获取每个目标CellType的细胞名，形成列表
cells_highlight <- lapply(target_types, function(ct) {
  WhichCells(combined, expression = CellType == ct)
})
names(cells_highlight) <- target_types

# 选取与target_types数目一致的颜色（可自定义、可自动配色）
# cols_highlight <- c("red", "blue", "green", "orange", "purple")  # 示例色板
# 或自动分配色:
cols_highlight <- scales::hue_pal()(length(target_types))

p1 <- DimPlot(
  combined,
  group.by = "CellType",
  cells.highlight = cells_highlight,
  cols.highlight = cols_highlight,
  cols = "gray90"
) +
  ggtitle("Highlight Selected CellTypes")
p1


In [ ]:
target_types <- c("Auco_9", "Auco_14", "Auco_4", "Auco_8", "Neural", "neuronal", "peptidergic", "Peptidocytes")

# 获取每个目标CellType的细胞名，形成列表
cells_highlight <- lapply(target_types, function(ct) {
  WhichCells(combined, expression = CellType == ct)
})
names(cells_highlight) <- target_types

# 选取与target_types数目一致的颜色（可自定义、可自动配色）
# cols_highlight <- c("red", "blue", "green", "orange", "purple")  # 示例色板
# 或自动分配色:
cols_highlight <- scales::hue_pal()(length(target_types))

p2 <- DimPlot(
  combined,
  group.by = "CellType",
  cells.highlight = cells_highlight,
  cols.highlight = cols_highlight,
  cols = "gray90"
) +
  ggtitle("Highlight Selected CellTypes")
p2
